# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
!pip install -q --upgrade pypdf gradio openai python-dotenv



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 517.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.4 MB/s eta 0:00:00


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [2]:
# Escribe tu código aquí

from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text

PDF_PATH = "attention_is_all_you_need.pdf"
texto = extract_text_from_pdf(PDF_PATH)
print(texto[:500])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [3]:
# Escribe tu código aquí

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

assert os.getenv("NVIDIA_API_KEY"), "Falta NVIDIA_API_KEY en el archivo .env"
print("NVIDIA_API_KEY cargada correctamente")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"

NVIDIA_API_KEY cargada correctamente


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [4]:
# Escribe tu código aquí

def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el paper "Attention Is All You Need" (Vaswani et al., 2017).

Responde las preguntas del usuario basándote ÚNICAMENTE en el documento delimitado a continuación.
No uses conocimiento externo ni inventes información.
Si la respuesta no está en el documento, responde exactamente:
"No encuentro esa información en el documento."

Responde en el mismo idioma en que te pregunte el usuario.

<documento>
{document_text}
</documento>"""


def history_to_messages(history):
    messages = []
    for turn in history:
        # Formato nuevo de Gradio: {"role": ..., "content": ...}
        if isinstance(turn, dict):
            content = turn.get("content", "")
            if isinstance(content, list):  # contenido en partes: nos quedamos con el texto
                content = " ".join(
                    part.get("text", "") if isinstance(part, dict) else str(part)
                    for part in content
                )
            messages.append({"role": turn["role"], "content": content})
        # Formato antiguo: [mensaje_usuario, respuesta_asistente]
        else:
            user_msg, assistant_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    return messages


def chat_con_documento(message, history, document_text):
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        max_tokens=4096,
        stream=True,
    )

    respuesta = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            respuesta += delta
            yield respuesta

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [6]:
# Escribe tu código aquí

import gradio as gr

document_text = extract_text_from_pdf(PDF_PATH)
n_paginas = len(PdfReader(PDF_PATH).pages)
print(f"Páginas: {n_paginas} | Caracteres extraídos: {len(document_text):,}")

ejemplos = [
    "¿Cuál es la arquitectura principal propuesta en el paper?",
    "¿Qué es el mecanismo de atención?",
    "¿Cuántas capas tiene el encoder del modelo base?",
]

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente de documentos: Attention Is All You Need",
    description="Haz preguntas sobre el paper. Las respuestas se basan únicamente en el documento.",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[[pregunta, document_text] for pregunta in ejemplos],
)

demo.launch(show_error=True)

Páginas: 15 | Caracteres extraídos: 39,600
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a5985e004b0532be5d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?

Las preguntas de la 1 a la 5 la responde de manera correcta de acuerdo al documento. Cuando se le pregunta la 6, da la siguiente respuesta: No encuentro esa información en el documento. Es decir que si se ciñe a la instrucción de solo basarse en el documento.







## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?

Si cada mensaje envía el documento completo no es para nada eficiente. Se paga en costo y latencia por tokens repetidos. Ademas, aunque quepan en el sistema, un contexto tan grande cada vez puede diluir la información relevante y empeorar la respuesta.

2. ¿Por qué este ejercicio todavía no implementa RAG?

Aqui no estamos fragmentando el documento para los prompts, si no que el documento completo se envía cada vez. No hay uso de RAG.

3. ¿Cómo comprobarían que la respuesta proviene del documento y no del conocimiento previo del modelo?

- Exigir que cada respuesta cite la página o una frase literal del documento, y verificar que esa cita exista en el texto.
- Hacer preguntas de control que no están en el paper (como GPT-4) y ver si responde “No encuentro esa información”.
- Modificar el documento y comprobar si la respuesta sigue al documento modificado o a lo que el modelo “sabe”.
- Comparar la respuesta con y sin el documento en el prompt.

4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

- Inyección de prompts indirecta: el documento puede contener instrucciones (“ignora lo anterior...”) que el modelo trate como órdenes.
- Fuga de información: datos sensibles o privados quedan en el prompt, en logs y en el proveedor de la API.
- Rotura de delimitadores: un documento con etiquetas como </documento> puede confundir la estructura del prompt.
- Costos y abuso: documentos enormes disparan los tokens consumidos o llenan la ventana de contexto.
- Contenido no confiable: el modelo puede repetir o amplificar información falsa o dañina del documento.

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)